# Sending Tool Results

Claude asked for a tool; now we **run it and send the answer back** — completing the request → `tool_use` → execute → `tool_result` → final answer loop.

**The `tool_result` block:**

| Field | Meaning |
|-------|---------|
| `tool_use_id` | Must match the `id` of the `tool_use` block it answers |
| `content` | Your function's output, **serialized as a string** |
| `is_error` | `True` if the tool raised/failed |

It goes inside a **user** message. Then the follow-up request must **still include the tool schema** — Claude needs it to interpret the tool references already in the history, even though you don't expect another call.

## Setup + the first call (steps 1–2 from last lesson)

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from datetime import datetime
from anthropic import Anthropic
from anthropic.types import ToolParam

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, message):
    content = message.content if hasattr(message, "content") else message
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, message):
    content = message.content if hasattr(message, "content") else message
    messages.append({"role": "assistant", "content": content})


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)

messages = []
add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

add_assistant_message(messages, response)   # preserve the multi-block turn
print("stop_reason:", response.stop_reason)

stop_reason: tool_use


## Step 3 — extract the tool call and run the function

The lesson grabs the tool block positionally as `response.content[1].input` (assumes a text block sits at index 0). That breaks if Claude returns *only* a tool_use block, so we scan for it by type instead. Then unpack the `input` dict as keyword args.

In [2]:
tool_use = next(block for block in response.content if block.type == "tool_use")

print("tool:  ", tool_use.name)
print("args:  ", tool_use.input)

# Call the real function with Claude's chosen arguments
tool_output = get_current_datetime(**tool_use.input)
print("output:", tool_output)

tool:   get_current_datetime
args:   {'date_format': '%H:%M:%S'}
output: 22:42:26


## Step 4 — send the result back as a `tool_result` block

It lives in a **user** message. `tool_use_id` ties it to the request; `content` is the output as a string; `is_error` is `False` on success. History is now: user question → assistant (text + tool_use) → user (tool_result).

In [3]:
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": tool_output,
                "is_error": False,
            }
        ],
    }
)

## Step 5 — the final request (schema still attached)

We include `tools=[...]` again so Claude can resolve the tool references in the history. It should now answer in natural language using the tool's output — and `stop_reason` returns to `"end_turn"`.

In [4]:
final = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print("stop_reason:", final.stop_reason)
print(get_text(final))

stop_reason: end_turn
The exact time is **22:42:26** (10:42:26 PM in 12-hour format).


## Loop complete

Claude answered with a real, current time it could never have known on its own. That's the whole custom-tool mechanic end to end.

Two things to carry forward:
- **Multiple tool calls in one turn** — a prompt like "what's 10+10 and 30+30?" can yield *several* `tool_use` blocks; you return one `tool_result` per `id` (order doesn't matter — the `tool_use_id` is what pairs them).
- Right now this is a **single, hard-coded round**. Next lessons generalize it into a loop (`stop_reason == "tool_use"` → run → resend, repeat) so Claude can chain tools — which is exactly what the reminder project needs (get now → add duration → set reminder).